# Getting Started with Arabic Prosody Helpers


This notebook serves as an interactive guide and tutorial for using the `arabic_prosody_helpers.py` module.


Built on top of the **pyarud** engine, this helper library provides high-level utilities for classical Arabic poetry analysis, including:


- Meter detection (ʿArūḍ) using a clean **U/\_ notation** (where `U` represents a short/mutaharrik syllable, and `_` represents a long/sākin syllable).

- Detailed diagnostic breakdowns of metrical deviations (_Zihāf_ and _ʿIlal_).

- Quality-of-fit scoring.

- Rhyme analysis (Rawiyy and Majra extraction).

- Sub-meter form classification (_Tamm_, _Majzūʾ_, _Mashtūr_, _Manhūk_).


---


## Prerequisites


To run this code, ensure you have **Python ≥ 3.12** and the underlying engine **pyarud ≥ 0.1.10** installed.



In [1]:
# pip install pyarud


We will start by confirming that the helper module is accessible in your current directory.



In [2]:
import os
import sys

# Ensure current directory is in the path so we can import our local file
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# Verify if arabic_prosody_helpers is importable
try:
    import arabic_prosody_helpers
    print("✓ Successfully imported arabic_prosody_helpers!")
except ImportError as e:
    print(f"✗ Failed to import: {e}. Ensure arabic_prosody_helpers.py is in the same folder.")

✓ Successfully imported arabic_prosody_helpers!



---


## Step 1: Meter Detection & Core Analysis


Let's begin by analyzing one of the most famous verses in classical Arabic poetry, written by Al-Mutanabbi.



In [3]:
from arabic_prosody_helpers import analyze_verse

# Al-Mutanabbi's famous verse in Al-Basit meter
sadr = "أَنَامُ مِلْءَ جُفُونِي عَنْ شَوَارِدِهَا"
ajuz = "وَيَسْهَرُ الْخَلْقُ جَرَّاهَا وَيَخْتَصِمُ"

# Analyze the single verse
result = analyze_verse(sadr, ajuz)

print(f"Detected Meter  : {result.meter}")
print(f"Combined Score  : {result.combined_score:.2f}")
print(f"Metrically Sound: {result.sadr.is_sound and result.ajuz.is_sound}")

Detected Meter  : baseet
Combined Score  : 1.00
Metrically Sound: True



---


## Step 2: Visualizing Metrical Alignment


The module provides tools to print plain-text alignments of syllables alongside classical diagnostics.



In [4]:
from arabic_prosody_helpers import visualize_verse_alignment, summarize_issues

# 1. Print visual alignment of patterns (U/_ notation)
visualize_verse_alignment(result)

# 2. Print a detailed metrical status report
print("\nMetrical Analysis Summary:")
print(summarize_issues(result))

Verse 1 Visual Alignment:
-----------------------------------------------------------------
Ṣadr text    : أَنَامُ مِلْءَ جُفُونِي عَنْ شَوَارِدِهَا
Ṣadr pattern : UU_UU_UUU_U_U_UU_UUU_
ʿAjuz text   : وَيَسْهَرُ الْخَلْقُ جَرَّاهَا وَيَخْتَصِمُ
ʿAjuz pattern: UU_UU_U_UU_U_U_UU_UUU_
-----------------------------------------------------------------

Metrical Analysis Summary:
Verse 1 — meter: البسيط (baseet) — score: 1.00
  ⚠ [Ṣadr] foot 1 (Hashw): valid zihāf «Qabadh» applied (canonical «UU_U_U_» → actual «UU_UU_»).
  ⚠ [Ṣadr] foot 2 (Hashw): valid zihāf «Khaban» applied (canonical «U_UU_» → actual «UUU_»).
  ⚠ [Ṣadr] foot 4 (ʿArūḍ): valid zihāf «Khaban» applied (canonical «U_UU_» → actual «UUU_»).
  ⚠ [ʿAjuz] foot 1 (Hashw): valid zihāf «Qabadh» applied (canonical «UU_U_U_» → actual «UU_UU_»).
  ⚠ [ʿAjuz] foot 4 (Ḍarb): valid zihāf «Khaban» applied (canonical «U_UU_» → actual «UUU_»).



---


## Step 3: Deeper Foot Analysis & Classical Tafāʿīl Mnemonics


Classical Arabic prosody relies on ten core rhythmic structures called _Tafāʿīl_ (e.g., _Mustafelon_, _Faelon_). The helper maps the binary engine output to standard Arabic mnemonics while naming the precise poetic license (_Zihāf_) applied.


Let's inspect each foot in the first hemistich (Ṣadr) to see what structural changes occurred:



In [5]:
from arabic_prosody_helpers import get_tafeela_mnemonic

print("Detailed Foot-by-Foot Analysis (Ṣadr):\n")

for foot in result.sadr.feet:
    # Resolve the expected canonical class name based on its index in Basit
    expected_class = "Mustafelon" if foot.foot_index in (0, 2) else "Faelon"

    # Retrieve the classical Arabic representation
    arabic_mnemonic = get_tafeela_mnemonic(expected_class, foot.zihaf_name)

    print(f"Foot {foot.foot_index + 1} ({foot.position_label}):")
    print(f"  • Expected Foot : {expected_class}")
    print(f"  • Observed Seg  : {foot.actual_segment}")
    print(f"  • Active Zihāf  : {foot.zihaf_name}")
    print(f"  • Mnemonic form : {arabic_mnemonic}")
    print(f"  • Quality Score : {foot.score:.2f}")
    print(f"  • Health Label  : {foot.health}")
    print()

Detailed Foot-by-Foot Analysis (Ṣadr):

Foot 1 (Hashw):
  • Expected Foot : Mustafelon
  • Observed Seg  : UU_UU_
  • Active Zihāf  : Qabadh
  • Mnemonic form : مُسْتَفْعِلُنْ (Qabadh)
  • Quality Score : 1.00
  • Health Label  : valid_zihaf

Foot 2 (Hashw):
  • Expected Foot : Faelon
  • Observed Seg  : UUU_
  • Active Zihāf  : Khaban
  • Mnemonic form : فَعِلُنْ
  • Quality Score : 1.00
  • Health Label  : valid_zihaf

Foot 3 (Hashw):
  • Expected Foot : Mustafelon
  • Observed Seg  : U_U_UU_
  • Active Zihāf  : Salim
  • Mnemonic form : مُسْتَفْعِلُنْ
  • Quality Score : 1.00
  • Health Label  : perfect

Foot 4 (ʿArūḍ):
  • Expected Foot : Faelon
  • Observed Seg  : UUU_
  • Active Zihāf  : Khaban
  • Mnemonic form : فَعِلُنْ
  • Quality Score : 1.00
  • Health Label  : valid_zihaf




---


## Step 4: Diagnostic Filters for Broken or Modified Feet


When dealing with large collections of lines, you may want to programmatically flag verses containing metrical mistakes or specific poetic licenses.



In [6]:
from arabic_prosody_helpers import get_broken_feet, get_zihaf_feet, filter_by_health

# Create a sample verse containing artificial metrical mistakes (missing letters)
broken_sadr = "أَنَامُ جُفُونِي عَنْ شَوَارِدِهَا" # Intentionally missing parts of the rhythm
broken_ajuz = "وَيَسْهَرُ الْخَلْقُ جَرَّاهَا وَيَخْتَصِمُ"

broken_result = analyze_verse(broken_sadr, broken_ajuz)

# Find issues programmatically
broken_feet_sadr = get_broken_feet(broken_result.sadr)
modified_feet_ajuz = get_zihaf_feet(broken_result.ajuz)

print(f"Broken feet in Sadr: {len(broken_feet_sadr)}")
for b_foot in broken_feet_sadr:
    print(f"  -> Foot {b_foot.foot_index + 1} failed. Expected pattern: {b_foot.expected_pattern}")

print(f"\nFeet with valid poetic licenses (Zihāf) in Ajuz: {len(modified_feet_ajuz)}")
for m_foot in modified_feet_ajuz:
    print(f"  -> Foot {m_foot.foot_index + 1} has active license: {m_foot.zihaf_name}")

Broken feet in Sadr: 2
  -> Foot 1 failed. Expected pattern: U_U_UU_
  -> Foot 2 failed. Expected pattern: U_UU_

Feet with valid poetic licenses (Zihāf) in Ajuz: 2
  -> Foot 1 has active license: Qabadh
  -> Foot 4 has active license: Khaban



---


## Step 5: Rhyme (Qāfiyah) Auditing


Poetry is defined not only by its meter but also by its rhyme consistency. The helper can extract the **Rawiyy** (الروي — the fundamental rhyming consonant) and the **Majra** (المجرى — its vowel) to ensure the poem remains uniform.



In [7]:
from arabic_prosody_helpers import extract_rawiyy, audit_rhyme_scheme

# Let's inspect a single hemistich end
rawiyy, majra = extract_rawiyy(ajuz)
print(f"For ending «{ajuz.split()[-1]}»:")
print(f"  • Rawiyy (الروي): '{rawiyy}'")
print(f"  • Majra (المجرى): '{majra}'")

# Now audit a multi-verse poem scheme
poem_verses = [
    ("أَنَامُ مِلْءَ جُفُونِي عَنْ شَوَارِدِهَا", "وَيَسْهَرُ الْخَلْقُ جَرَّاهَا وَيَخْتَصِمُ"),
    ("وَمَا انْتِفَاعُ أَخِي الدُّنْيَا بِعَيْنِهِ", "إِذَا اسْتَوَتْ عِنْدَهُ الأَنْوَارُ وَالظُّلَمُ")
]

rhyme_report = audit_rhyme_scheme(poem_verses)
print("\nRhyme Scheme Audit Report:")
print(f"  • Consistent?     : {rhyme_report['consistent']}")
print(f"  • Dominant Rawiyy : '{rhyme_report['dominant_rawiyy']}'")
print(f"  • Dominant Majra  : '{rhyme_report['dominant_majra']}'")

if rhyme_report['issues']:
    print("  • Violations found:")
    for issue in rhyme_report['issues']:
        print(f"    - {issue}")
else:
    print("  • No rhyme scheme violations found!")

For ending «وَيَخْتَصِمُ»:
  • Rawiyy (الروي): 'م'
  • Majra (المجرى): 'Damma'

Rhyme Scheme Audit Report:
  • Consistent?     : True
  • Dominant Rawiyy : 'م'
  • Dominant Majra  : 'Damma'
  • No rhyme scheme violations found!



---


## Step 6: Determining Verse Layout Sub-Meters


poetics allow variations such as _Majzūʾ_ (dropping a foot from both halves of a verse) or _Mashtūr_ (halving the verse structure entirely). You can automatically analyze the overall structure using `classify_sub_meter`.



In [8]:
from arabic_prosody_helpers import analyze_poem, classify_sub_meter

# Re-analyze our short poem
poem_result = analyze_poem(poem_verses)

# Determine the structural layout variation
sub_meter_info = classify_sub_meter(poem_result)

print("Sub-Meter Classification Info:")
print(f"  • Primary Meter      : {sub_meter_info['meter']}")
print(f"  • Poetic Form Layout : {sub_meter_info['layout']}")
print(f"  • Structural Details : {sub_meter_info['explanation']}")
print(f"  • Avg Active Feet    : {sub_meter_info['average_active_feet']} / {sub_meter_info['total_canonical_feet']}")

Sub-Meter Classification Info:
  • Primary Meter      : baseet
  • Poetic Form Layout : Tamm
  • Structural Details : Complete structural form (Tamm).
  • Avg Active Feet    : 4.0 / 4



---


## Step 7: Displaying Reference Meter Tables (Prosodic Lexicon)


The module ships with pre-compiled reference dictionaries containing classical definitions and boundary restrictions for key meters like _Tawīl_, _Basīt_, _Kāmil_, _Wāfir_, _Ramal_, _Mutaqārib_, _Mutadārak_, _Rajaz_, and _Khafīf_.


You can render formatted guides directly for an AI prompt or human reference:



In [9]:
from arabic_prosody_helpers import get_meter_table, resolve_meter_key

# Resolve various alias inputs to canonical keys (e.g., 'البسيط', 'basit')
canonical_key = resolve_meter_key("الكامل")
print(f"Resolved Key: {canonical_key}\n")

# Fetch and print the fully compiled guide
guide_text = get_meter_table(canonical_key)
print(guide_text)

Resolved Key: kamil

═══════════════════════════════════════════════════════
البحر: الكَامِل  (al-Kāmil)
الضرب: مُتَفَاعِلُنْ
═══════════════════════════════════════════════════════

التفعيلات الأساسية:
  • مُتَفَاعِلُنْ: UUU_UU_

الشطر الكامل:
  بالرموز:  UUU_UU_ | UUU_UU_ | UUU_UU_
  عروضياً: مُتَفَاعِلُنْ | مُتَفَاعِلُنْ | مُتَفَاعِلُنْ
  التكرار:  مُتَفَاعِلُنْ — تتكرر ثلاث مرات في كل شطر

الزحافات الجائزة:
  • الإضمار على مُتَفَاعِلُنْ: تسكين التاء (الثاني المتحرك) → مُتْفَاعِلُنْ (U_U_UU_)
  • الوقص على مُتَفَاعِلُنْ: حذف التاء (الثاني المتحرك) → مُفَاعِلُنْ (UU_UU_)
  • الخزل على مُتَفَاعِلُنْ: إضمار + طي (تسكين التاء + حذف الألف) → مُتْفَعِلُنْ (U_UUUU_)

صور العروض والضرب:
  • عروض: مُتَفَاعِلُنْ — صحيحة (UUU_UU_)
    ضرب:  مُتَفَاعِلُنْ — صحيح (UUU_UU_)
  • عروض: مُتَفَاعِلُنْ — صحيحة (UUU_UU_)
    ضرب:  مُتَفَاعِلْ — مقطوع (UUU_U_)
  • عروض: مُتَفَاعِلُنْ — صحيحة (UUU_UU_)
    ضرب:  فَعْلُنْ — أحذ مضمر (U_U_)
  • عروض: فَعِلُنْ — حذاء (UUU_)
    ضرب:  فَعِلُنْ — أحذ (UUU_)
 


---


⚑ **Critical Operational Insight: Tashkeel (Vocalisation) Sensitivity**

The underlying analyzer relies heavily on Arabic diacritics (_Tashkeel_) to construct the sound sequences required for mapping patterns.


- **If your input text lacks complete diacritics**, the processing system will make structural errors, resulting in heavily penalized match scores or inaccurate meter detections.

- **Best Practice:** Always run raw Arabic input through a diacritization model (like _Shakkala_ or _Mishkal_) prior to passing the text to `analyze_poem` or `analyze_verse`.


## Step 8: Analyzing a Complete Poem (List of Tuples)

Often, you will want to analyze a full poem rather than a single standalone verse. In python, we represent a poem as a list of tuples, where each tuple is a `(sadr, ajuz)` pair.

The library offers two primary ways to analyze a poem:
1. `analyze_poem()`: Returns a rich, nested `PoemResult` object that holds detailed metadata, candidate meters, and a list of `VerseResult` structures.
2. `analyze_poem_to_dict()`: Executes the same analysis, outputs an immediate, user-friendly printout showing accuracy percentages line-by-line, and returns the entire tree as a standard Python dictionary (highly useful for JSON serialization or API endpoints).

### 1. Analyzing using the `PoemResult` object

Let's analyze a 3-verse poem written by Al-Mutanabbi.



In [10]:
from arabic_prosody_helpers import analyze_poem, summarize_issues, analyze_poem_to_dict


In [11]:
lyrics = [
    (
        "كَامِنٌ فِي الرِّمَالِ يَخْفَى صَوْتِي",
        "حَيْثُ لَا يُدْرَكُ الجَلِيلُ بُعَدَا",
    ),
    (
        "فِي تَدَاخُلِ الأُغْنِيَاتِ يَضِيعُ",
        "صَوْتُ مَنْ كَانَ فِي الصَّدَى مِثْلَ عَهْدَا",
    ),
    (
        "وَامِضًا كَالبُرُوقِ يَبْرُقُ صَوْتٌ",
        "سَاطِعًا فِي المَسَامِعِ بِالأَمَانِ",
    ),
    (
        "حَاضِرًا جَلْجَلَ السَّمِيعِ يُنَادِي",
        "كَالضِّيَاءِ المُنِيرِ يَبْقَى لَحْنَا",
    ),
    (
        "يَنْحَنِي الصَّوْتُ خَافِضاً جَهْرَهُ",
        "حِينَ يَعْلُو صَدًى لَهُ غَرِيبَا",
    ),
    (
        "كَالخُضُوعِ الجَلِيلِ لِرَبِّهِ",
        "يَتَرَاجَى الصَّدَى بِكُلِّ حَبِيبَا",
    ),
    (
        "بَيْنَ هَمْسٍ وَجَهْرِهِ مَدْيُهُ اتَّسَعَا",
        "وَالنَّغَمْ يَرْتَقِي كَمَاءِ السَّمَاءِ",
    ),
    (
        "جَمْعُهُ اللُّطْفَ وَالجَلَالَ انْبِسَاطٌ",
        "لَا يُخَافُ الجُمُودُ فِي البِنَاءِ",
    ),
    (
        "فَنُّ مَزْجِ الأَصْوَاتِ حِينَ تَوَزَّعَتْ",
        "زَهْرَةٌ فِي فَضَاءِ السَّمْعِ قَدْ أَشْرَقَتْ",
    ),
    (
        "صَانِعُ المَزْجِ إِنْ أَحَاطَ بِأَبْعَادٍ",
        "يَسْتَحِقُّ الجَمَالَ ذَاكَ الخِفَاقُ",
    ),
    (
        "يَبْلُغُ الصَّوْتُ ذِرْوَةَ الحُسْنِ وَالكَمَالْ",
        "حِينَ يُحْكَمُ مَزْجُهُ جَمِيعَا",
    ),
    (
        "مِنْ هَمُوسٍ إِلَى الجَلَجِلَةِ انْتِقَالُهُ",
        "فَارْفَعَنْ بِالتَّوَازُنِ المُطِيعَا",
    ),
]

In [17]:
poem = [
    ("آذَنَتْنَا بِبَيْنِهَا أَسْمَاءُ", "رُبَّ ثَاوٍ يُمَلُّ مِنْهُ الثَّوَاءُ"),
    ("آذَنَتْنَا بِبَيْنِهَا ثُمَّ وَلَّتْ", "لَيْتَ شِعْرِي مَتَى يَكُونُ اللِّقَاءُ"),
    ("بَعْدَ عَهْدٍ لَنَا بِبُرْقَةِ شَمَّاءَ", "فَأَدْنَى دِيَارِهَا الْخَلْصَاءُ"),
    ("فَالْمُحَيَّاةُ فَالصِّفَاحُ فَأَعْنَا", "قُ فِتَاقٍ فَعَاذِبٌ فَالْوَفَاءُ"),
    ("فَرِيَاضُ الْقَطَا فَأَوْدِيَةُ الشُّرْ", "بَبِ فَالشُّعْبَتَانِ فَالْأَبْلَاءُ"),
    ("لَا أَرَى مَنْ عَهِدْتُ فِيهَا فَأَبْكِي", "الْيَوْمَ دَلْهًا وَمَا يَرُدُّ الْبُكَاءُ"),
    ("وَبِعَيْنَيْكَ أَوْقَدَتْ هِنْدٌ النَّا", "رَ أَخِيرًا تُلْوِي بِهَا الْعَلْيَاءُ"),
    ("أَوْقَدَتْهَا بَيْنَ الْعَقِيقِ فَشَخْصَيْ", "نِ بِعُودٍ كَمَا يَلُوحُ الضِّيَاءُ"),
    ("فَتَنَوَّرْتُ نَارَهَا مِنْ بَعِيدٍ", "بِخَزَازٍ هَيْهَاتَ مِنْكَ الصِّلَاءُ"),
    ("غَيْرَ أَنِّي قَدْ أَسْتَعِينُ عَلَى الْهَمْ", "مِ إِذَا خَفَّ بِالثَّوِيِّ النَّجَاءُ"),
    ("بِزَفُوفٍ كَأَنَّهَا هِقْلَةٌ أُمْ", "مُ رِئَالٍ دَوِيَّةٌ سَقْفَاءُ"),
    ("آنَسَتْ نَبْأَةً وَأَفْزَعَهَا الْقَنْ", "نَاصُ عَصْرًا وَقَدْ دَنَا الْإِمْسَاءُ"),
    ("فَتَرَى خَلْفَهَا مِنَ الرَّجْعِ وَالْوَقْ", "عِ مَنِينًا كَأَنَّهُ إِهْبَاءُ"),
    ("وَطِرَاقًا مِنْ خَلْفِهِنَّ طِرَاقٌ", "سَاقِطَاتٌ أَلْوَتْ بِهَا الصَّحْرَاءُ"),
    ("أَتَلَهَّى بِهَا الْهَوَاجِرَ إِذْ كُلْ", "لُ ابْنِ هَمٍّ بَلِيَّةٌ عَمْيَاءُ")
]

In [18]:

# Run the complete poem analysis
poem_result = analyze_poem_to_dict(poem, meter_name='khafif')


Verse 1: «آذَنَتْنَا بِبَيْنِهَا أَسْمَاءُ | رُبَّ ثَاوٍ يُمَلُّ مِنْهُ الثَّوَاءُ» — Accuracy: 86.00%
Verse 2: «آذَنَتْنَا بِبَيْنِهَا ثُمَّ وَلَّتْ | لَيْتَ شِعْرِي مَتَى يَكُونُ اللِّقَاءُ» — Accuracy: 93.00%
Verse 3: «بَعْدَ عَهْدٍ لَنَا بِبُرْقَةِ شَمَّاءَ | فَأَدْنَى دِيَارِهَا الْخَلْصَاءُ» — Accuracy: 85.00%
Verse 4: «فَالْمُحَيَّاةُ فَالصِّفَاحُ فَأَعْنَا | قُ فِتَاقٍ فَعَاذِبٌ فَالْوَفَاءُ» — Accuracy: 100.00%
Verse 5: «فَرِيَاضُ الْقَطَا فَأَوْدِيَةُ الشُّرْ | بَبِ فَالشُّعْبَتَانِ فَالْأَبْلَاءُ» — Accuracy: 100.00%
Verse 6: «لَا أَرَى مَنْ عَهِدْتُ فِيهَا فَأَبْكِي | الْيَوْمَ دَلْهًا وَمَا يَرُدُّ الْبُكَاءُ» — Accuracy: 87.00%
Verse 7: «وَبِعَيْنَيْكَ أَوْقَدَتْ هِنْدٌ النَّا | رَ أَخِيرًا تُلْوِي بِهَا الْعَلْيَاءُ» — Accuracy: 92.00%
Verse 8: «أَوْقَدَتْهَا بَيْنَ الْعَقِيقِ فَشَخْصَيْ | نِ بِعُودٍ كَمَا يَلُوحُ الضِّيَاءُ» — Accuracy: 100.00%
Verse 9: «فَتَنَوَّرْتُ نَارَهَا مِنْ بَعِيدٍ | بِخَزَازٍ هَيْهَاتَ مِنْكَ الصِّلَاءُ» — Accuracy: 100.00%
Verse 10: «غَيْرَ أَ

In [19]:
lyrics = [
    (
        "يَكْتَمِي اللَّحْنُ فِي الرِّمَالِ كَأَنْ لَا",
        "حِينَ يَخْفَى الكَنْزُ فِي الثَّرَى بَعِيدَا",
    ),
    (
        "يَنْحَنِي الصَّوْتُ خَافِضاً جَهْرَهُ",
        "لَيْسَ يُدْرَى المَدَى بِهِ لِجَدِيدَا",
    ),
    (
        "يَلْمَعُ الصَّوْتُ عَابِراً كَسَنَا البَرْقِ",
        "حِينَ يَشْفِي صَدَاهُ كُلَّ لِسَانٍ",
    ),
    (
        "يَظْهَرُ الصَّوْتُ جَلْيَةً فِي السَّمَاعِ",
        "عِنْدَمَا يَعْلُو نَقْعُهُ لِلْحَنِينَا",
    ),
    (
        "يَنْحَنِي الصَّوْتُ خَافِضاً جَهْرَهُ",
        "حِينَ يَعْلُو صَدًى لَهُ غَرِيبَا",
    ),
    (
        "يَخْضَعُ الجَهْرُ لِلْعُلَا فَيَنِي",
        "كَمَا يَحْنُو الحَبِيبُ لِلطِّيبَا",
    ),
    (
        "يَتَّسِعْ مَدْيُهُ بَيْنَ الهَمْسِ وَالجَهْرِ",
        "فَيُضِيءُ الحَيَاةَ كَالضِّيَاءِ",
    ),
    (
        "مَا بَدَا الصَّوْتُ فِي فَضَاءِ النَّقَاءِ",
        "إِلَّا كَالشِّفَاءِ لِلصَّدَى بِنَاءً",
    ),
    (
        "حِينَ يُحْسِنْ مُزِيجَهُ الصَّانِعُ",
        "يَزْدَهِي الفَنُّ كَالأَشْرَقْ",
    ),
    (
        "فِي فَضَاءِ السَّمْعِ تَنْتَظِمُ الأَصْوَاتُ",
        "كُلَّمَا يَعْشَقُ المُزِيجُ وَيَتْلُو",
    ),
    (
        "بَلَغَ الصَّوْتُ ذُرْوَةَ الحُسْنِ لَمَّا",
        "أُحْكِمَ المَزْجُ فِي النَّسِيجِ جَمِيعَا",
    ),
    (
        "فَاعْتَدَلْ مَدْيُهُ مِنَ الهَمْسِ لِلجَلْجَلَةِ",
        "مِثْلَ نَغْمٍ يَصْعَدُ الرَّفِيعَا",
    ),
]

In [20]:
# Run the complete poem analysis
poem_result = analyze_poem_to_dict(lyrics, meter_name='khafif')


Verse 1: «يَكْتَمِي اللَّحْنُ فِي الرِّمَالِ كَأَنْ لَا | حِينَ يَخْفَى الكَنْزُ فِي الثَّرَى بَعِيدَا» — Accuracy: 87.00%
Verse 2: «يَنْحَنِي الصَّوْتُ خَافِضاً جَهْرَهُ | لَيْسَ يُدْرَى المَدَى بِهِ لِجَدِيدَا» — Accuracy: 100.00%
Verse 3: «يَلْمَعُ الصَّوْتُ عَابِراً كَسَنَا البَرْقِ | حِينَ يَشْفِي صَدَاهُ كُلَّ لِسَانٍ» — Accuracy: 93.00%
Verse 4: «يَظْهَرُ الصَّوْتُ جَلْيَةً فِي السَّمَاعِ | عِنْدَمَا يَعْلُو نَقْعُهُ لِلْحَنِينَا» — Accuracy: 100.00%
Verse 5: «يَنْحَنِي الصَّوْتُ خَافِضاً جَهْرَهُ | حِينَ يَعْلُو صَدًى لَهُ غَرِيبَا» — Accuracy: 100.00%
Verse 6: «يَخْضَعُ الجَهْرُ لِلْعُلَا فَيَنِي | كَمَا يَحْنُو الحَبِيبُ لِلطِّيبَا» — Accuracy: 77.00%
Verse 7: «يَتَّسِعْ مَدْيُهُ بَيْنَ الهَمْسِ وَالجَهْرِ | فَيُضِيءُ الحَيَاةَ كَالضِّيَاءِ» — Accuracy: 74.00%
Verse 8: «مَا بَدَا الصَّوْتُ فِي فَضَاءِ النَّقَاءِ | إِلَّا كَالشِّفَاءِ لِلصَّدَى بِنَاءً» — Accuracy: 87.00%
Verse 9: «حِينَ يُحْسِنْ مُزِيجَهُ الصَّانِعُ | يَزْدَهِي الفَنُّ كَالأَشْرَقْ» — Accuracy: 90.00%
Verse 1


### 2. Quick CLI analysis with Dictionary output

If you want an automatic terminal printout and a serialized dictionary structure, use `analyze_poem_to_dict`.


In [16]:

from arabic_prosody_helpers import analyze_poem_to_dict

# This function will automatically print each verse with its calculated accuracy percentage
poem_dict = analyze_poem_to_dict(lyrics, meter_name='khafif')

# We can inspect the returned dictionary structure
print("\n--- Serialized Dictionary (Sample Fields) ---")
print(f"Dictionary Type: {type(poem_dict)}")
print(f"Meter          : {poem_dict['meter']} ({poem_dict['meter_arabic']})")
print(f"Overall Score  : {poem_dict['overall_score']:.4f}")
print(f"Sound Poem?    : {poem_dict['is_metrically_sound']}")
print(f"First Verse Sadr Pattern: {poem_dict['verses'][0]['sadr']['pattern']}")


Verse 1: «يَكْتَمِي اللَّحْنُ فِي الرِّمَالِ كَأَنْ لَا | حِينَ يَخْفَى الكَنْزُ فِي الثَّرَى بَعِيدَا» — Accuracy: 87.00%
Verse 2: «يَنْحَنِي الصَّوْتُ خَافِضاً جَهْرَهُ | لَيْسَ يُدْرَى المَدَى بِهِ لِجَدِيدَا» — Accuracy: 100.00%
Verse 3: «يَلْمَعُ الصَّوْتُ عَابِراً كَسَنَا البَرْقِ | حِينَ يَشْفِي صَدَاهُ كُلَّ لِسَانٍ» — Accuracy: 93.00%
Verse 4: «يَظْهَرُ الصَّوْتُ جَلْيَةً فِي السَّمَاعِ | عِنْدَمَا يَعْلُو نَقْعُهُ لِلْحَنِينَا» — Accuracy: 100.00%
Verse 5: «يَنْحَنِي الصَّوْتُ خَافِضاً جَهْرَهُ | حِينَ يَعْلُو صَدًى لَهُ غَرِيبَا» — Accuracy: 100.00%
Verse 6: «يَخْضَعُ الجَهْرُ لِلْعُلَا فَيَنِي | كَمَا يَحْنُو الحَبِيبُ لِلطِّيبَا» — Accuracy: 77.00%
Verse 7: «يَتَّسِعْ مَدْيُهُ بَيْنَ الهَمْسِ وَالجَهْرِ | فَيُضِيءُ الحَيَاةَ كَالضِّيَاءِ» — Accuracy: 74.00%
Verse 8: «مَا بَدَا الصَّوْتُ فِي فَضَاءِ النَّقَاءِ | إِلَّا كَالشِّفَاءِ لِلصَّدَى بِنَاءً» — Accuracy: 87.00%
Verse 9: «حِينَ يُحْسِنْ مُزِيجَهُ الصَّانِعُ | يَزْدَهِي الفَنُّ كَالأَشْرَقْ» — Accuracy: 90.00%
Verse 1

In [21]:
from arabic_prosody_feedback import analyze_and_report

In [23]:
analyze_and_report(poem, meter_name='khafif')

Verse 1: «آذَنَتْنَا بِبَيْنِهَا أَسْمَاءُ | رُبَّ ثَاوٍ يُمَلُّ مِنْهُ الثَّوَاءُ» — Accuracy: 86.00%
Verse 2: «آذَنَتْنَا بِبَيْنِهَا ثُمَّ وَلَّتْ | لَيْتَ شِعْرِي مَتَى يَكُونُ اللِّقَاءُ» — Accuracy: 93.00%
Verse 3: «بَعْدَ عَهْدٍ لَنَا بِبُرْقَةِ شَمَّاءَ | فَأَدْنَى دِيَارِهَا الْخَلْصَاءُ» — Accuracy: 85.00%
Verse 4: «فَالْمُحَيَّاةُ فَالصِّفَاحُ فَأَعْنَا | قُ فِتَاقٍ فَعَاذِبٌ فَالْوَفَاءُ» — Accuracy: 100.00%
Verse 5: «فَرِيَاضُ الْقَطَا فَأَوْدِيَةُ الشُّرْ | بَبِ فَالشُّعْبَتَانِ فَالْأَبْلَاءُ» — Accuracy: 100.00%
Verse 6: «لَا أَرَى مَنْ عَهِدْتُ فِيهَا فَأَبْكِي | الْيَوْمَ دَلْهًا وَمَا يَرُدُّ الْبُكَاءُ» — Accuracy: 87.00%
Verse 7: «وَبِعَيْنَيْكَ أَوْقَدَتْ هِنْدٌ النَّا | رَ أَخِيرًا تُلْوِي بِهَا الْعَلْيَاءُ» — Accuracy: 92.00%
Verse 8: «أَوْقَدَتْهَا بَيْنَ الْعَقِيقِ فَشَخْصَيْ | نِ بِعُودٍ كَمَا يَلُوحُ الضِّيَاءُ» — Accuracy: 100.00%
Verse 9: «فَتَنَوَّرْتُ نَارَهَا مِنْ بَعِيدٍ | بِخَزَازٍ هَيْهَاتَ مِنْكَ الصِّلَاءُ» — Accuracy: 100.00%
Verse 10: «غَيْرَ أَ

({'meter': 'khafeef',
  'meter_arabic': 'الخفيف',
  'total_verses': 15,
  'verses': [{'verse_index': 0,
    'sadr': {'text': 'آذَنَتْنَا بِبَيْنِهَا أَسْمَاءُ',
     'pattern': 'UUUU_U_UU_UU_U_U_U_',
     'feet': [{'foot_index': 0,
       'expected_pattern': 'U_UU_U_',
       'actual_segment': 'UUUU_U_',
       'canonical_pattern': 'U_UU_U_',
       'score': 0.4,
       'status': 'broken',
       'zihaf_name': None,
       'health': 'broken',
       'position_label': 'Hashw'},
      {'foot_index': 1,
       'expected_pattern': 'UU_UU_',
       'actual_segment': 'UU_UU_',
       'canonical_pattern': 'UU_U_U_',
       'score': 1.0,
       'status': 'ok',
       'zihaf_name': 'Qabadh',
       'health': 'valid_zihaf',
       'position_label': 'Hashw'},
      {'foot_index': 2,
       'expected_pattern': 'U_UU_',
       'actual_segment': 'U_U_U',
       'canonical_pattern': 'U_UU_',
       'score': 0.26,
       'status': 'broken',
       'zihaf_name': None,
       'health': 'broken',
       